In [1]:
import re
import pyodbc
import pandas as pd

In [ ]:
def parse_ruby_hash_list(text):
    blocks = re.findall(r'\{([^}]+)\}', text)
    records = []

    for block in blocks:
        record = {}
        pairs = re.findall(r':(\w+)=>"?([^",}]+)"?', block)

        for key, value in pairs:
            value = value.strip()
            if re.fullmatch(r'\d+', value):
                value = int(value)
            elif re.fullmatch(r'\d+\.\d+', value):
                value = float(value)
            record[key] = value

        if record:
            records.append(record)

    return pd.DataFrame(records)


# Load file
with open(r"../../SourceFiles/Task1/task1_d.json", 'r', encoding='utf-8') as f:
    text = f.read()

df = parse_ruby_hash_list(text)

In [46]:
df['id'] = df['id'].astype(str)
df.head(5)

,id,title,author,genre,publisher,year,price
0,10292064894005717421,Look Homeward,Prof. Teressa Kautzer,Humor,Brill Publishers,2010,$87.25
1,13029911509625386835,The Yellow Meads of Asphodel,Domingo Weimann,Reference book,Sams Publishing,2018,$31.99
2,12880574241579659568,A Catskill Eagle,Dayle Orn,Comic/Graphic Novel,Apress,2011,€5.99
3,13301315742612799364,Der Richter und sein Henker,Elias von Kolb,Tall tale,Centaurus Verlag,1995,$75.00
4,16372759776603821045,After Many a Summer Dies the Swan,Carter Legros,Metafiction,University of Minnesota Press,2004,$52.0


In [47]:
df.dtypes

id           object
title        object
author       object
genre        object
publisher    object
year          int64
price        object
dtype: object

In [49]:
import pyodbc
import pandas as pd

# --- Convert types ---
df['id'] = df['id'].astype(str)
df['year'] = df['year'].apply(lambda x: int(x) if pd.notnull(x) else None)

# --- Connect ---
conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=DESKTOP-5KGKU83;"
    "DATABASE=Itransition;"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

cursor = conn.cursor()
cursor.fast_executemany = True


# --- Insert ---
data = list(
    df[['id','title','author','genre','publisher','year','price']]
    .itertuples(index=False, name=None)
)

cursor.executemany("""
    INSERT INTO tasks.books_raw (id, title, author, genre, publisher, year, price)
    VALUES (?, ?, ?, ?, ?, ?, ?)
""", data)

conn.commit()
cursor.close()
conn.close()

print("Data loaded ✅")

Data loaded ✅
